# SCI1070 — Semaine 8 : le gradient de politique, et sa variance

| Volet | Ce que vous y faites |
| --- | --- |
| A | REINFORCE avec et sans référence, et **la variance du gradient mesurée** |
| B | A2C vectorisé sur CartPole, trois termes de perte, dix quantités journalisées |

**Ce que le laboratoire n'implémente pas, et pourquoi.** A3C n'est pas au programme :
les processus multiples sont limités dans un carnet partagé. Le volet B utilise des
**environnements vectorisés synchrones**. **La section 8.10 se lit, se discute et ne
s'exécute pas**, et c'est une décision assumée.

> **Je ne vous donne aucun chiffre à atteindre pour A2C sur CartPole.** J'attends une
> mesure **relative et reproductible** : votre nombre de pas à la résolution, sur au
> moins trois graines, avec son intervalle, comparé à votre propre REINFORCE sur les
> **mêmes** graines.

In [ ]:
import importlib.util
import subprocess
import sys

for paquet, module in [("gymnasium[classic-control]>=1.0", "gymnasium"), ("torch", "torch")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", paquet], check=True)
if importlib.util.find_spec("sci1070") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "git+https://github.com/FaycalZM/drl-course-teluq.git#subdirectory=code"],
                   check=True)

import collections
from dataclasses import dataclass
from typing import Optional

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from sci1070.protocole import intervalle, se_recouvrent
from sci1070.courbes import bande, tracer_bande
from sci1070.graines import fixer_graines

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GRAINES = [0, 1, 2]
print("périphérique :", device)

---
# Volet A : REINFORCE avec et sans référence

**Objectif.** Produire **une** figure : la variance du gradient de politique au cours
de l'entraînement, deux courbes, avec et sans référence, sur les mêmes graines.

### Étape 1 — REINFORCE, avec un seul interrupteur

In [ ]:
class ReseauPolitique(nn.Module):
    def __init__(self, n_obs, n_actions, n_cache=128):
        super().__init__()
        ...        # À COMPLÉTER : sorties BRUTES, le softmax vient au moment du tirage

    def forward(self, x):
        return ...        # À COMPLÉTER : un seul appel

### Étape 2 — la référence

La **moyenne glissante** des retours actualisés. **Deux lignes suffisent.**

In [ ]:
class MoyenneGlissante:
    """La référence du volet. Deux lignes, et c'est tout ce qu'il faut."""

    def __init__(self):
        self.somme, self.n = 0.0, 0

    def ajouter(self, x):
        ...        # À COMPLÉTER : accumuler

    @property
    def valeur(self):
        return ...        # À COMPLÉTER : la moyenne, et le cas n = 0


m = MoyenneGlissante()
for x in (1.0, 2.0, 3.0):
    m.ajouter(x)
print("moyenne glissante :", m.valeur)

### Étape 3, le cœur du volet — isoler le gradient de politique

La manœuvre de la section 8.6 : rétropropager **ce terme seul** avec
`retain_graph=True`, lire les gradients **avant** de rétropropager l'entropie, puis
laisser l'optimiseur avancer sur la somme.

In [ ]:
def statistiques_du_gradient(reseau):
    """Les trois statistiques, lues sur les gradients actuellement accumulés."""
    g = torch.cat([p.grad.flatten() for p in reseau.parameters() if p.grad is not None])
    return {"variance": float(g.var()), "norme_l2": float(g.norm()),
            "max_abs": float(g.abs().max())}


print("statistiques_du_gradient prête")

### Les deux vérifications arithmétiques, **avant** de conclure

In [ ]:
gamma, horizon = 0.99, 10
attendu = sum(gamma ** i for i in range(horizon))
print(f"la référence doit converger vers {attendu:.4f}")
assert abs(attendu - 9.5618) < 1e-3

In [ ]:
def reinforce(reference, graine, max_pas=120_000, gamma=0.99, horizon=10):
    """Rend (pas_a_la_resolution, courbe_variance, courbe_retour)."""
    ...        # À COMPLÉTER : REINFORCE, l'indicateur `reference` étant le SEUL interrupteur

### Étape 4 — la figure

Les deux courbes de variance en **échelle logarithmique**, parce que l'écart attendu
est de deux à trois ordres de grandeur.

In [ ]:
resultats = {ref: [reinforce(ref, g) for g in GRAINES] for ref in (False, True)}

fig, ax = plt.subplots(figsize=(8, 5))
for ref, etiquette in [(False, "sans référence"), (True, "avec référence")]:
    courbes = ...        # À COMPLÉTER : les courbes de variance, une par graine
    n = min(len(c) for c in courbes)
    tracer_bande(ax, np.arange(n), [c[:n] for c in courbes], etiquette=etiquette)
ax.set_yscale("log")
ax.set_xlabel("mise à jour"); ax.set_ylabel("variance du gradient de politique")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

In [ ]:
for ref, etiquette in [(False, "sans référence"), (True, "avec référence")]:
    pas = [r[0] for r in resultats[ref]]
    moyenne, bas, haut = intervalle(pas)
    print(f"{etiquette:18s} {moyenne:9.0f} pas   [{bas:.0f}, {haut:.0f}]")

print("\nles deux intervalles se recouvrent :",
      se_recouvrent([r[0] for r in resultats[False]], [r[0] for r in resultats[True]]))

**Répondez par écrit à deux questions.** L'écart que vous mesurez est-il du même
ordre que celui annoncé en section 8.6 ? Et la version avec référence résout-elle
le problème **en moins de pas** ? **Ce n'est pas la même question**, et les deux
réponses peuvent diverger : réduire la variance d'un estimateur n'accélère la
convergence que si la variance était le facteur limitant.

---
# Volet B : A2C vectorisé sur CartPole

### Étape 1 — les environnements

**Attention aux deux attributs** : ce sont `single_observation_space` et
`single_action_space` qui décrivent **un** environnement.

In [ ]:
N_ENVS = 16

envs = gym.vector.SyncVectorEnv(
    [lambda: gym.make("CartPole-v1") for _ in range(N_ENVS)]
)
taille_obs = envs.single_observation_space.shape[0]   # 4, et non (16, 4)
n_actions = int(envs.single_action_space.n)           # 2, et non (16,)

print("observation d'UN environnement :", taille_obs, "| actions :", n_actions)
print("observation du LOT             :", envs.observation_space.shape)
obs, info = envs.reset(seed=0)
print("après reset :", obs.shape)
envs.close()

### Étape 2 — la cible à \(N\) pas, la fonction la plus délicate du volet

Cette fonction vous est donnée entière, parce que son intérêt est dans ce qu'elle
**évite**, pas dans sa frappe.

In [ ]:
@dataclass
class ExperienceRepliee:
    etat: np.ndarray
    action: int
    recompense_actualisee: float
    dernier_etat: Optional[np.ndarray]


def preparer_lot(lot, reseau, gamma: float, n_pas: int, device):
    """(etats, actions, cibles) a partir d'un lot de transitions premier-dernier."""
    etats, actions, cibles = [], [], []
    derniers_etats, idx_non_terminaux = [], []

    for i, exp in enumerate(lot):
        etats.append(exp.etat)
        actions.append(exp.action)
        cibles.append(exp.recompense_actualisee)
        if exp.dernier_etat is not None:        # l'episode continue
            idx_non_terminaux.append(i)
            derniers_etats.append(exp.dernier_etat)

    etats_t = torch.as_tensor(np.asarray(etats), dtype=torch.float32, device=device)
    actions_t = torch.as_tensor(actions, dtype=torch.int64, device=device)
    cibles = np.asarray(cibles, dtype=np.float32)

    if derniers_etats:
        derniers_t = torch.as_tensor(np.asarray(derniers_etats),
                                     dtype=torch.float32, device=device)
        with torch.no_grad():                   # le critique est INTERROGE, pas entraine ici
            v_derniers = reseau(derniers_t)[1].squeeze(-1).cpu().numpy()
        cibles[idx_non_terminaux] += (gamma ** n_pas) * v_derniers

    return etats_t, actions_t, torch.as_tensor(cibles, device=device)


print("preparer_lot est fournie : lisez-la, ne la réécrivez pas")

### Étape 3 — la boucle

**N'oubliez ni le `detach()` ni le `eps` d'Adam : ce sont les deux lignes qui
décident si ce volet fonctionne.**

In [ ]:
class ReseauA2C(nn.Module):
    """Tronc commun, deux têtes : la politique et la valeur."""

    def __init__(self, n_obs, n_actions, n_cache=128):
        super().__init__()
        ...        # À COMPLÉTER : le tronc, puis deux têtes. forward rend un COUPLE (logits, valeur).

    def forward(self, x):
        ...        # À COMPLÉTER : le couple

In [ ]:
def a2c(graine, n_envs=N_ENVS, n_pas=4, gamma=0.99, coef_entropie=0.01,
        ecretage=0.5, sans_entropie=False, sans_ecretage=False, max_pas=200_000):
    """Rend (pas_a_la_resolution, journal) où journal porte les dix quantités."""
    ...        # À COMPLÉTER : la boucle complète. Les trois interrupteurs servent à l'ablation de l'étape 5.

### Étape 4 — les dix quantités à journaliser

A2C n'a **aucune** perte dont la valeur vous dise si les choses se passent bien :
ces courbes sont votre seule fenêtre.

In [ ]:
journaux = {g: a2c(graine=g)[1] for g in GRAINES}

quantites = ["avantage", "valeur", "recompense", "perte_politique", "perte_valeur",
             "perte_entropie", "perte_totale", "grad_norme_l2", "grad_max_abs",
             "grad_variance"]

fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for ax, quantite in zip(axes.ravel(), quantites):
    courbes = ...        # À COMPLÉTER : la bande sur les trois graines
    n = min(len(c) for c in courbes)
    tracer_bande(ax, np.arange(n), [c[:n] for c in courbes])
    ax.set_title(quantite); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Étape 5 — l'ablation, et elle est rapide

In [ ]:
ablations = {
    "complet": dict(),
    "sans entropie": dict(sans_entropie=True),
    "sans écrêtage": dict(sans_ecretage=True),
    "un seul environnement": dict(n_envs=1),
}

...        # À COMPLÉTER : une ligne de tableau par ablation, sur les MÊMES graines

### Diagnostic facultatif, et je vous le recommande vivement

Une exécution saine reste **en dessous de \(10^{-3}\)**. **Retenez cette quantité :
elle est le sujet de la semaine 10.**

In [ ]:
def kl_avant_apres(logits_avant, logits_apres):
    """La divergence KL d'un pas d'optimisation, sur un lot."""
    p_avant = F.softmax(logits_avant, dim=1)
    log_avant = F.log_softmax(logits_avant, dim=1)
    log_apres = F.log_softmax(logits_apres, dim=1)
    return float((p_avant * (log_avant - log_apres)).sum(dim=1).mean())


# Un déplacement minuscule, et un déplacement brutal, pour fixer l'échelle.
torch.manual_seed(0)
avant = torch.randn(64, 2)
print(f"pas minuscule : KL = {kl_avant_apres(avant, avant + 0.01):.2e}  (sain)")
print(f"pas brutal    : KL = {kl_avant_apres(avant, avant + 1.00):.2e}  (alerte)")

**Attendu du volet B.** La courbe de récompense sur au moins trois graines avec
intervalle, les dix quantités journalisées, le tableau de vos trois ablations, et la
comparaison avec votre REINFORCE du volet A **sur les mêmes graines**.

> **Mon message pour ce laboratoire.** Le volet A mesure une variance et le volet B
> construit un algorithme, et c'est dans cet ordre que je vous demande de les faire.
> Vous n'avez, dans A2C, **aucune** perte dont la valeur vous dise si les choses se
> passent bien. **Sans les repères du volet A, le volet B est une boîte noire qui
> monte ou ne monte pas.**